**Microsoft Defender for Cloud** (formerly Azure Security Center + Azure Defender) is Azure's unified cloud security posture management (CSPM) and cloud workload protection platform (CWPP). It gives you:

- **Visibility** — continuous assessment of your security posture across Azure, on-premises, and multi-cloud
- **Hardening** — prioritised recommendations to reduce attack surface
- **Protection** — threat detection and alerts for workloads at runtime

| Capability | Tool | AWS equivalent |
|---|---|---|
| Security posture management | Defender for Cloud (CSPM) | AWS Security Hub |
| Workload threat protection | Defender for Cloud (CWPP / Defender plans) | Amazon GuardDuty |
| Compliance assessment | Regulatory compliance dashboard | AWS Security Hub standards |
| Vulnerability scanning | Defender for Servers / Containers | Amazon Inspector |
| SIEM integration | Microsoft Sentinel | AWS Security Lake + Amazon Detective |

## Cloud Security Posture Management (CSPM)

The CSPM capabilities are available in two tiers:

| Tier | Cost | Capabilities |
|---|---|---|
| **Foundational CSPM** | Free | Secure Score, basic recommendations, asset inventory |
| **Defender CSPM** | Per resource/hour | Attack path analysis, cloud security graph, data-sensitive posture, agentless scanning, EASM integration |

### Secure Score

**Secure Score** is a single percentage that summarises your security posture across your subscription or management group. It is calculated from **security recommendations** grouped into **security controls**:

```
Security Controls (e.g. "Remediate vulnerabilities")
  └── Recommendations (e.g. "System updates should be installed on machines")
        └── Affected resources
```

Each control has a **maximum score** contribution. Completing all recommendations in a control awards its full points. The overall Secure Score is the ratio of your current points to the maximum possible.

### Security recommendations

Defender for Cloud continuously evaluates your resources against security best practices and generates **recommendations** when resources deviate. Each recommendation includes:
- Affected resources
- Remediation steps (often a one-click fix or an ARM template)
- **Severity** (High / Medium / Low)
- Score impact

Examples:
- *MFA should be enabled on accounts with owner permissions*
- *Storage accounts should restrict network access*
- *Managed disks should be encrypted with customer-managed keys*
- *Public network access on Azure SQL Database should be disabled*

### Attack path analysis

**Attack path analysis** (Defender CSPM) builds a **cloud security graph** by modelling relationships between resources, identities, and network paths. It identifies chains of misconfigurations that could allow an attacker to move laterally from a publicly exposed resource to a sensitive data store — and presents the highest-risk paths for prioritised remediation.

### Regulatory compliance

The **Regulatory Compliance** dashboard maps your resources against compliance standards:

| Standard | Audience |
|---|---|
| **Microsoft Cloud Security Benchmark (MCSB)** | Azure default baseline |
| **CIS Azure Foundations Benchmark** | General Azure hardening |
| **NIST SP 800-53** | US federal |
| **PCI DSS** | Payment card industry |
| **ISO 27001** | Information security management |
| **SOC 2** | Service organisations |
| **HIPAA** | US healthcare |

For each control in the standard, Defender for Cloud shows which Azure resources are compliant and which are not, with direct links to remediation recommendations.

## Microsoft Defender Plans (CWPP)

**Defender plans** are per-resource-type threat protection services enabled in a subscription. Each plan adds advanced threat detection, vulnerability assessment, and alerts for a specific resource type:

| Plan | Protects | Key capabilities |
|---|---|---|
| **Defender for Servers** | Azure VMs, on-prem via Arc | EDR (Microsoft Defender for Endpoint integration), vulnerability assessment, file integrity monitoring, JIT VM access, adaptive application controls |
| **Defender for SQL** | Azure SQL DB, SQL MI, SQL on VM | Vulnerability assessment, Advanced Threat Protection (SQL injection detection, anomalous access) |
| **Defender for Storage** | Blob, Files, ADLS | Malware scanning, sensitive data discovery, anomalous access alerts |
| **Defender for Containers** | AKS, ACR, on-prem K8s | Image vulnerability scanning, runtime threat detection, Kubernetes audit log analysis |
| **Defender for App Service** | App Service, Functions | Anomalous activity detection, dangling DNS detection |
| **Defender for Key Vault** | Key Vault | Anomalous access patterns, exfiltration attempts |
| **Defender for Resource Manager** | ARM operations | Detect suspicious ARM API usage, lateral movement |
| **Defender for DNS** | Azure DNS | DNS exfiltration, malicious domain queries |
| **Defender for APIs** | Azure API Management | API discovery, anomalous API usage |

### Defender for Servers — key features

#### Just-in-Time (JIT) VM Access

**JIT VM Access** closes management ports (RDP 3389, SSH 22) when not in use and opens them on demand for a limited time window:

```
Normal state:   NSG rule blocks RDP/SSH inbound — port closed
Request access: User requests JIT access (time + source IP)
                Defender updates NSG to allow access for 1–24 hours from specific IP
Access expires: NSG rule revoked automatically — port closed again
```

Every JIT access request is logged in the Activity Log — providing a full audit trail.

#### Adaptive Application Controls

Machine learning analyses processes running on a group of VMs and recommends an **allowlist** of legitimate applications. Any process not on the allowlist triggers an alert. This prevents malware execution even if an attacker gains access.

#### File Integrity Monitoring (FIM)

Monitors changes to OS files, Windows registry keys, and application files — alerting when critical system files are modified (indicator of compromise).

### Defender for Containers — key features

| Feature | Description |
|---|---|
| **Image scanning (ACR)** | Scan container images pushed to ACR for OS and package vulnerabilities using Microsoft Defender Vulnerability Management |
| **Runtime threat detection** | Monitor running containers for suspicious behaviour — privileged container access, crypto mining, shell spawned in container |
| **Kubernetes audit log analysis** | Detect anomalous Kubernetes API calls — exposed dashboards, secrets enumeration, role binding changes |
| **Agentless scanning** | Scan AKS node VMs for vulnerabilities without deploying an agent |

## Security Alerts and Incidents

When Defender for Cloud detects a threat, it generates a **security alert**:

| Property | Description |
|---|---|
| **Severity** | High / Medium / Low / Informational |
| **MITRE ATT&CK tactic** | Maps to MITRE ATT&CK framework — Initial Access, Execution, Persistence, Lateral Movement, Exfiltration, etc. |
| **Affected resource** | Which VM, storage account, Key Vault, etc. was targeted |
| **Evidence** | Log entries, IP addresses, user accounts involved |
| **Remediation steps** | Specific actions to take |

### Security incidents

Related alerts are automatically **correlated into incidents** — a single attack campaign that spans multiple resources is grouped together, making it easier to understand the full scope of an attack rather than treating each alert in isolation.

### Alert suppression rules

Define **suppression rules** to mute known-benign alerts — for example, suppress alerts for a specific management script that runs regularly on a known IP.

### Workflow automation

**Workflow automation** triggers a Logic App when an alert or recommendation fires — enabling automated responses such as:
- Send a Teams or Slack notification
- Create a ticket in ServiceNow
- Isolate a compromised VM automatically

## Azure Policy and Defender for Cloud

Defender for Cloud recommendations are built on **Azure Policy**. Each recommendation corresponds to one or more Azure Policy definitions. When you enable a Defender plan or a compliance standard, Azure Policy initiatives are assigned to your subscription.

### Security policies

You can customise which recommendations are active by:
- **Disabling** a recommendation for your subscription (if not applicable)
- **Exempting** specific resources from a recommendation
- **Assigning additional standards** from the regulatory compliance catalogue

### Continuous export

**Continuous export** streams Defender for Cloud data to:
- **Log Analytics workspace** — query security data with KQL alongside other Azure Monitor logs
- **Event Hub** — stream to a SIEM like Microsoft Sentinel, Splunk, or IBM QRadar

## Microsoft Sentinel

**Microsoft Sentinel** is a cloud-native SIEM (Security Information and Event Management) and SOAR (Security Orchestration, Automated Response) built on Log Analytics.

```
Data connectors  →  Log Analytics workspace  →  Analytics rules  →  Incidents
                                                                          ↓
                                                                  Automation (SOAR)
```

### Key components

| Component | Description |
|---|---|
| **Data connectors** | Ingest logs from Azure services, Microsoft 365, third-party firewalls, Syslog |
| **Analytics rules** | KQL-based detection rules that create incidents when triggered |
| **Incidents** | Grouped alerts for investigation by security analysts |
| **Hunting queries** | Proactive threat hunting with KQL across all ingested data |
| **Workbooks** | Security dashboards built on Azure Monitor Workbooks |
| **Automation rules / Playbooks** | Logic App playbooks triggered on incidents for automated response |
| **UEBA** | User and Entity Behaviour Analytics — ML-based anomaly detection |
| **Threat intelligence** | Import threat intel feeds (IP/domain reputation) for enrichment |

### Defender for Cloud → Sentinel integration

Connect Defender for Cloud to Sentinel via the data connector — all Defender alerts and recommendations flow into Sentinel as incidents for unified investigation and response.

### Sentinel vs Defender for Cloud

| | Defender for Cloud | Microsoft Sentinel |
|---|---|---|
| Scope | Azure + multi-cloud workloads | Enterprise-wide (all data sources) |
| Focus | Posture + workload threat protection | SIEM/SOAR — correlate, investigate, respond |
| Query language | KQL (recommendations + alerts) | KQL (full log analytics) |
| Typical user | Cloud security team | SOC analysts |

## Multi-Cloud and Hybrid Support

Defender for Cloud extends beyond Azure:

| Environment | Integration |
|---|---|
| **On-premises servers** | Azure Arc — enrol servers into Azure management plane; Defender for Servers protects them |
| **AWS** | Native AWS connector — ingest AWS Security Hub findings; assess EC2, RDS, EKS |
| **GCP** | Native GCP connector — assess GCP Compute Engine, GKE |

The Secure Score, recommendations, and regulatory compliance dashboard cover all connected environments in a single view — giving security teams a single pane of glass across hybrid and multi-cloud estates.

## Working with Defender for Cloud via Python

In [ ]:
# pip install azure-identity azure-mgmt-security

In [ ]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.security import SecurityCenter

credential      = DefaultAzureCredential()
subscription_id = "<your-subscription-id>"
resource_group  = "my-rg"

security = SecurityCenter(credential, subscription_id)

In [ ]:
# Get overall Secure Score for the subscription
for score in security.secure_scores.list():
    if score.display_name == "Azure Defender":
        current = score.score.current if score.score else 0
        maximum = score.score.max if score.score else 0
        pct     = (current / maximum * 100) if maximum else 0
        print(f"Secure Score: {current:.2f} / {maximum:.2f}  ({pct:.1f}%)")

# List Secure Score controls
for ctrl in security.secure_score_controls.list():
    if ctrl.score and ctrl.score.max and ctrl.score.max > 0:
        pct = ctrl.score.current / ctrl.score.max * 100
        print(f"  {ctrl.display_name:<55} {ctrl.score.current:.1f}/{ctrl.score.max:.1f} ({pct:.0f}%)")

In [ ]:
# List high-severity recommendations
high_sev = [
    r for r in security.assessments.list(f"/subscriptions/{subscription_id}")
    if hasattr(r, 'status') and r.status and r.status.code == "Unhealthy"
]

print(f"Unhealthy assessments: {len(high_sev)}")
for rec in high_sev[:10]:
    name = rec.display_name if hasattr(rec, 'display_name') else rec.name
    print(f"  [{rec.status.code}] {name}")

In [ ]:
# List active security alerts
alerts = list(security.alerts.list())
print(f"Total active alerts: {len(alerts)}")

for alert in sorted(alerts, key=lambda a: a.time_generated_utc or '', reverse=True)[:10]:
    severity     = alert.alert_type or "unknown"
    resource     = alert.compromised_entity or "unknown"
    intent       = alert.intent or "unknown"
    time_fired   = alert.time_generated_utc
    print(f"  [{alert.alert_severity}] {alert.alert_display_name:<50} resource: {resource}  intent: {intent}")

In [ ]:
from azure.mgmt.security.models import (
    JitNetworkAccessPolicy, JitNetworkAccessPolicyVirtualMachine,
    JitNetworkAccessPortRule
)

vm_resource_id = (
    f"/subscriptions/{subscription_id}/resourceGroups/{resource_group}"
    "/providers/Microsoft.Compute/virtualMachines/my-vm"
)

# Enable JIT access policy on a VM — close RDP and SSH by default
jit_policy = security.jit_network_access_policies.create_or_update(
    resource_group,
    "eastus",           # location
    "my-jit-policy",
    JitNetworkAccessPolicy(
        virtual_machines=[
            JitNetworkAccessPolicyVirtualMachine(
                id=vm_resource_id,
                ports=[
                    JitNetworkAccessPortRule(
                        number=3389,           # RDP
                        protocol="TCP",
                        allowed_source_address_prefix="*",
                        max_request_access_duration="PT3H"  # max 3 hours
                    ),
                    JitNetworkAccessPortRule(
                        number=22,             # SSH
                        protocol="TCP",
                        allowed_source_address_prefix="*",
                        max_request_access_duration="PT3H"
                    )
                ]
            )
        ]
    )
)
print(f"JIT policy created: {jit_policy.name}")

In [ ]:
from azure.mgmt.security.models import Pricing

# List current Defender plan status for all resource types
for pricing in security.pricings.list(f"/subscriptions/{subscription_id}"):
    status = pricing.pricing_tier  # Free or Standard
    print(f"  {pricing.name:<30} {status}")

In [ ]:
# Enable Defender for Storage plan
security.pricings.update(
    scope=f"/subscriptions/{subscription_id}",
    pricing_name="StorageAccounts",
    pricing=Pricing(pricing_tier="Standard")
)
print("Defender for Storage enabled")

# Enable Defender for Key Vault
security.pricings.update(
    scope=f"/subscriptions/{subscription_id}",
    pricing_name="KeyVaults",
    pricing=Pricing(pricing_tier="Standard")
)
print("Defender for Key Vault enabled")

In [ ]:
# List regulatory compliance standards and their pass/fail status
for std in security.regulatory_compliance_standards.list():
    passed  = std.passed_controls  or 0
    failed  = std.failed_controls  or 0
    skipped = std.skipped_controls or 0
    total   = passed + failed + skipped
    pct     = (passed / total * 100) if total else 0
    print(f"  {std.name:<40} {passed}/{total} controls ({pct:.0f}% compliant)")

## Summary

| Concept | Key Takeaway |
|---|---|
| Defender for Cloud | Unified CSPM + CWPP — posture management and workload threat protection |
| Foundational CSPM | Free — Secure Score, basic recommendations, asset inventory |
| Defender CSPM | Paid — attack path analysis, cloud security graph, agentless scanning |
| Secure Score | % of security best practices met; calculated from controls and recommendations |
| Security recommendations | Actionable findings with severity, remediation steps, and score impact |
| Attack path analysis | Identifies chains of misconfigurations leading from public exposure to sensitive data |
| Regulatory compliance | Maps resources to CIS, NIST, PCI DSS, ISO 27001, HIPAA, SOC 2 |
| Defender plans | Per-resource-type threat protection — Servers, SQL, Storage, Containers, Key Vault, etc. |
| JIT VM Access | Close RDP/SSH by default; open on-demand for limited time + specific IP |
| Adaptive Application Controls | ML-based process allowlist — alerts on unknown executables |
| File Integrity Monitoring | Alert on changes to OS/config files — indicator of compromise |
| Defender for Containers | Image scanning in ACR + runtime threat detection in AKS |
| Security alerts | Threat detections mapped to MITRE ATT&CK; correlated into incidents |
| Workflow automation | Logic App triggered on alert/recommendation — automated response |
| Continuous export | Stream alerts + recommendations to Log Analytics or Event Hub / SIEM |
| Microsoft Sentinel | Cloud-native SIEM/SOAR on Log Analytics — correlate, hunt, respond |
| Defender for Cloud → Sentinel | Connect via data connector — unified investigation across enterprise |
| Multi-cloud | Arc for on-prem; native AWS + GCP connectors — single Secure Score |
| MCSB | Microsoft Cloud Security Benchmark — Azure default compliance baseline |